# 1. LangGraph Basics — Your First Graph

## What is LangGraph?

**LangChain** chains components in a mostly straight line: prompt → LLM → parser.
**LangGraph** lets you build your LLM application as a **graph**: a set of steps (**nodes**)
connected by arrows (**edges**), all reading and writing one shared **state**.

Why does that matter? Because real applications need things a straight chain can't do:

| Need | LangGraph feature |
|---|---|
| "If X, do A, otherwise do B" | Conditional edges (notebook 2) |
| Remember previous conversations | Checkpointers / persistence (notebook 3) |
| Let the LLM decide which tool to call, in a loop | Cycles + ToolNode (notebook 4) |
| Pause and wait for a human to approve | `interrupt()` (notebook 5) |
| Several specialized agents cooperating | Multi-agent graphs (notebook 6) |

## The 4 core concepts

1. **State** — a shared dictionary (defined with `TypedDict`) that flows through the graph. Every node reads it and returns updates to it.
2. **Node** — a plain Python function: takes the state, returns a `dict` with the keys it wants to update.
3. **Edge** — connects nodes. `START → a → b → END`.
4. **Graph** — you build it with `StateGraph`, then `.compile()` it into a runnable app.

## Warm-up: a graph with zero LLMs

LangGraph is just Python — no API key needed to understand the mechanics.
Watch how each node returns **only the keys it changes** (a *partial update*), not the whole state.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class CoffeeState(TypedDict):
    order: str
    ground: bool
    brewed: bool
    message: str


def grind_beans(state: CoffeeState):
    print(f"Grinding beans for: {state['order']}")
    return {"ground": True}                      # partial update


def brew(state: CoffeeState):
    print(f"Brewing... (beans ground: {state['ground']})")
    return {"brewed": True}


def serve(state: CoffeeState):
    return {"message": f"Here is your {state['order']}. Enjoy!"}


builder = StateGraph(CoffeeState)
builder.add_node("grind_beans", grind_beans)
builder.add_node("brew", brew)
builder.add_node("serve", serve)

builder.add_edge(START, "grind_beans")
builder.add_edge("grind_beans", "brew")
builder.add_edge("brew", "serve")
builder.add_edge("serve", END)

coffee_app = builder.compile()
coffee_app.invoke({"order": "flat white", "ground": False, "brewed": False, "message": ""})

Grinding beans for: flat white
Brewing... (beans ground: True)


{'order': 'flat white',
 'ground': True,
 'brewed': True,
 'message': 'Here is your flat white. Enjoy!'}

## Real-life example: customer support email pipeline

A small business gets support emails. Every email should be processed the same way:

1. **extract** — pull out the customer's name and the actual problem
2. **draft_reply** — write a polite reply addressing the problem
3. **quality_check** — score the draft 1–10 so bad drafts can be flagged

This is a *linear pipeline* — the simplest useful LangGraph shape. Each step is one node.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env in this repo stores the key as GOOGLE_API_KEY_1 — normalize it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm.invoke("Say 'ready' if you can hear me.").content

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class EmailState(TypedDict):
    email: str          # raw incoming email (input)
    customer_name: str  # filled by extract
    problem: str        # filled by extract
    draft: str          # filled by draft_reply
    quality_score: str  # filled by quality_check

In [ ]:
def extract(state: EmailState):
    prompt = f'''Extract from this support email.
Reply in exactly this format, nothing else:
NAME: <customer name, or "unknown">
PROBLEM: <one-sentence summary of the problem>

Email:
{state["email"]}'''
    text = llm.invoke(prompt).content
    name = text.split("NAME:")[1].split("PROBLEM:")[0].strip()
    problem = text.split("PROBLEM:")[1].strip()
    return {"customer_name": name, "problem": problem}


def draft_reply(state: EmailState):
    prompt = f'''Write a short, warm support reply (max 120 words).
Customer name: {state["customer_name"]}
Their problem: {state["problem"]}'''
    return {"draft": llm.invoke(prompt).content}


def quality_check(state: EmailState):
    prompt = f'''Rate this support reply from 1-10 for politeness and usefulness.
Reply with ONLY the number.

{state["draft"]}'''
    return {"quality_score": llm.invoke(prompt).content.strip()}

In [ ]:
builder = StateGraph(EmailState)
builder.add_node("extract", extract)
builder.add_node("draft_reply", draft_reply)
builder.add_node("quality_check", quality_check)

builder.add_edge(START, "extract")
builder.add_edge("extract", "draft_reply")
builder.add_edge("draft_reply", "quality_check")
builder.add_edge("quality_check", END)

app = builder.compile()

In [ ]:
# Visualize the graph (needs internet for mermaid rendering — safe to skip)
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render image, here is the mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

In [ ]:
result = app.invoke({
    "email": '''Hi, I'm Priya. I ordered a blue backpack last Tuesday (order #4521)
and it still hasn't shipped. I need it before my trip on Friday. Can you help?'''
})

print("Customer:", result["customer_name"])
print("Problem :", result["problem"])
print("Score   :", result["quality_score"])
print("\n--- Draft reply ---\n")
print(result["draft"])

## Key takeaways

- A node is **just a function**: `state in → dict of updates out`.
- Nodes return **partial updates** — LangGraph merges them into the shared state.
- `START` and `END` are special markers, not nodes you write.
- `.compile()` turns the builder into an app with `.invoke()` / `.stream()`.

A linear graph like this is no smarter than a LangChain chain — the power starts in
**notebook 2**, where the graph *chooses its own path*.